In [ ]:
# Workspace initialization
import os

base_dir = "project"
subfolders = [
    "scanned_raw_folder",  
    "sorted_folder",       
    "final_folder"         
]

for folder in subfolders:
    dir_path = os.path.join(base_dir, folder)
    os.makedirs(dir_path, exist_ok=True)

In [ ]:
# Step 1: OCR Processing Routine
import os
import pytesseract
from PIL import Image

pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
scanned_dir = os.path.join("project", "scanned_raw_folder")
dataset = []
valid_extensions = (".png", ".jpg", ".jpeg", ".tiff", ".bmp")
all_files = [f for f in os.listdir(scanned_dir) if f.lower().endswith(valid_extensions)]
total_count = len(all_files)
print(f" Starting Tesseract OCR on {total_count} scanned images...\n")

for i, filename in enumerate(all_files, start=1):
    file_path = os.path.join(scanned_dir, filename)
    try:
        image = Image.open(file_path)
        ocr_text = pytesseract.image_to_string(image)
        dataset.append({
            "filename": filename,
            "text": ocr_text
        })
        print(f"[{i}/{total_count}]  OCR Processed: {filename}")
    except Exception as e:
        print(f"[{i}/{total_count}]  Failed OCR on {filename}: {e}")
print(f"\n Step 1 Complete! Total images loaded into dataset: {len(dataset)}")

 Starting Tesseract OCR on 100 scanned images...

[1/100]  OCR Processed: 00000831.jpg
[2/100]  OCR Processed: 0000958516.jpg
[3/100]  OCR Processed: 0000972342.jpg
[4/100]  OCR Processed: 0001166781.jpg
[5/100]  OCR Processed: 0001251167.jpg
[6/100]  OCR Processed: 0001414986.jpg
[7/100]  OCR Processed: 0011987564.jpg
[8/100]  OCR Processed: 10005191.jpg
[9/100]  OCR Processed: 10021964.jpg
[10/100]  OCR Processed: 10032894.jpg
[11/100]  OCR Processed: 10044882_10044909.jpg
[12/100]  OCR Processed: 10103478_10103482.jpg
[13/100]  OCR Processed: 10127889_10127890.jpg
[14/100]  OCR Processed: 10162340_10162342.jpg
[15/100]  OCR Processed: 10164982.jpg
[16/100]  OCR Processed: 10167656.jpg
[17/100]  OCR Processed: 10176813.jpg
[18/100]  OCR Processed: 10223631.jpg
[19/100]  OCR Processed: 10230391.jpg
[20/100]  OCR Processed: 10232045_10232047.jpg
[21/100]  OCR Processed: 10235850.jpg
[22/100]  OCR Processed: 10375498.jpg
[23/100]  OCR Processed: 10377432.jpg
[24/100]  OCR Processed: 103

In [ ]:
import os
import shutil
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

# Training Corpus Setup
training_data = [
    ("invoice bill amount due subtotal tax invoice_number due_date total price item qty quantity payment unit price vendor invoice_to ship_to receipt", "Invoice"),
    ("agreement contract parties non disclosure confidential hereby witnesseth section affiliate agreement terms conditions governing law signature witness liabilities", "Agreement"),
    ("balance sheet income statement cash flow financial statement fiscal year revenue net profit operating expenses assets liabilities annual report 10-k audit balance", "Financial Report")
]
train_texts = [text for text, label in training_data]
train_labels = [label for text, label in training_data]
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')
X_train = vectorizer.fit_transform(train_texts)
model = MultinomialNB()
model.fit(X_train, train_labels)
print(" Machine Learning Model Successfully Trained!\n")

# File Classification & Sorting Loop
sorted_base_dir = os.path.join("project", "sorted_folder")
raw_base_dir = os.path.join("project", "scanned_raw_folder")
print(" Sorting files based on ML predictions...\n")
for doc in dataset:
    doc_text = doc["text"] if doc["text"].strip() else "empty document"
    doc_vector = vectorizer.transform([doc_text])
    predicted_category = model.predict(doc_vector)[0]
    doc["category"] = predicted_category
    target_dir = os.path.join(sorted_base_dir, predicted_category)
    os.makedirs(target_dir, exist_ok=True)
    src_path = os.path.join(raw_base_dir, doc["filename"])
    dst_path = os.path.join(target_dir, doc["filename"])
    if os.path.exists(src_path):
        shutil.copy(src_path, dst_path)
    print(f" [{doc['filename']}]  --> Classified as: {predicted_category}")

print(f"\n Step 2 Complete! All {len(dataset)} files classified and sorted successfully.")

 Machine Learning Model Successfully Trained!

 Sorting files based on ML predictions...

 [00000831.jpg]  --> Classified as: Agreement
 [0000958516.jpg]  --> Classified as: Financial Report
 [0000972342.jpg]  --> Classified as: Financial Report
 [0001166781.jpg]  --> Classified as: Agreement
 [0001251167.jpg]  --> Classified as: Agreement
 [0001414986.jpg]  --> Classified as: Agreement
 [0011987564.jpg]  --> Classified as: Agreement
 [10005191.jpg]  --> Classified as: Agreement
 [10021964.jpg]  --> Classified as: Financial Report
 [10032894.jpg]  --> Classified as: Agreement
 [10044882_10044909.jpg]  --> Classified as: Financial Report
 [10103478_10103482.jpg]  --> Classified as: Financial Report
 [10127889_10127890.jpg]  --> Classified as: Agreement
 [10162340_10162342.jpg]  --> Classified as: Agreement
 [10164982.jpg]  --> Classified as: Agreement
 [10167656.jpg]  --> Classified as: Agreement
 [10176813.jpg]  --> Classified as: Financial Report
 [10223631.jpg]  --> Classified as: Fi

In [18]:
sorted_base = os.path.join("project", "sorted_folder")

print("Sorting Summary:")
for folder in os.listdir(sorted_base):
    folder_path = os.path.join(sorted_base, folder)
    if os.path.isdir(folder_path):
        file_count = len(os.listdir(folder_path))
        print(f" {folder}: {file_count} files")

Sorting Summary:
 Agreement: 31 files
 Financial Report: 32 files
 Invoice: 37 files


In [ ]:
import json
import requests
def extract_fields_fast(text):
    """Sends document OCR text to local Ollama (llama3) and returns structured JSON."""
    prompt = f"""Extract key information from this document text and return ONLY valid JSON:
    - document_id (e.g., invoice number, contract ID, or report name)
    - total_amount (monetary values or totals mentioned)
    - date (transaction or effective dates)
    - vendor_or_parties (companies or entities involved)
    - summary (1 concise sentence describing the file)
    Document Text:
    {text[:2000]}"""
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3",
            "prompt": prompt,
            "format": "json",
            "stream": False
        },
        timeout=60
    )
    res_json = response.json()
    return json.loads(res_json["response"])
print(" Starting Fast Local LLM Extraction via Ollama (llama3)...\n")
for i, doc in enumerate(dataset, start=1):
    try:
        doc["metadata"] = extract_fields_fast(doc["text"])
        print(f"[{i}/{len(dataset)}]  Extracted Metadata: {doc['filename']}")
    except Exception as e:
        doc["metadata"] = {
            "document_id": "N/A",
            "total_amount": "N/A",
            "date": "N/A",
            "vendor_or_parties": "N/A",
            "summary": f"Parsing Error: {str(e)}"
        }
        print(f"[{i}/{len(dataset)}] Fallback on {doc['filename']}: {e}")
print("\n Step 3 Complete! Local LLM Extraction finished for all documents.")

 Starting Fast Local LLM Extraction via Ollama (llama3)...

[1/100]  Extracted Metadata: 00000831.jpg
[2/100]  Extracted Metadata: 0000958516.jpg
[3/100]  Extracted Metadata: 0000972342.jpg
[4/100]  Extracted Metadata: 0001166781.jpg
[5/100]  Extracted Metadata: 0001251167.jpg
[6/100]  Extracted Metadata: 0001414986.jpg
[7/100]  Extracted Metadata: 0011987564.jpg
[8/100]  Extracted Metadata: 10005191.jpg
[9/100]  Extracted Metadata: 10021964.jpg
[10/100]  Extracted Metadata: 10032894.jpg
[11/100]  Extracted Metadata: 10044882_10044909.jpg
[12/100]  Extracted Metadata: 10103478_10103482.jpg
[13/100]  Extracted Metadata: 10127889_10127890.jpg
[14/100]  Extracted Metadata: 10162340_10162342.jpg
[15/100]  Extracted Metadata: 10164982.jpg
[16/100]  Extracted Metadata: 10167656.jpg
[17/100]  Extracted Metadata: 10176813.jpg
[18/100]  Extracted Metadata: 10223631.jpg
[19/100]  Extracted Metadata: 10230391.jpg
[20/100]  Extracted Metadata: 10232045_10232047.jpg
[21/100]  Extracted Metadata: 10

In [ ]:
import os
import json
import pandas as pd
final_dir = os.path.join("project", "final_folder")
os.makedirs(final_dir, exist_ok=True)
# 1. STRUCTURED DATA EXPORT (CSV & JSON)
structured_records = []
for doc in dataset:
    meta = doc.get("metadata", {})
    record = {
        "filename": doc.get("filename", ""),
        "category": doc.get("category", "Uncategorized"),
        "document_id": meta.get("document_id", "N/A"),
        "total_amount": meta.get("total_amount", "N/A"),
        "date": meta.get("date", "N/A"),
        "vendor_or_parties": meta.get("vendor_or_parties", "N/A"),
        "summary": meta.get("summary", "N/A")
    }
    structured_records.append(record)
json_path = os.path.join(final_dir, "extracted_data.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(structured_records, f, indent=4)
df_records = pd.DataFrame(structured_records)
csv_path = os.path.join(final_dir, "extracted_data.csv")
df_records.to_csv(csv_path, index=False)
print(" Structured Deliverables Successfully Exported!")
print(f"  JSON: {json_path}")
print(f"  CSV:  {csv_path}\n")

# 2. AUTOMATED ANOMALY AUDITING ENGINE
anomalies = []
for doc in dataset:
    filename = doc.get("filename", "")
    text = doc.get("text", "")
    meta = doc.get("metadata", {})
    category = doc.get("category", "Uncategorized")
    issues = []
    if len(text.strip()) < 50:
        issues.append("Empty or unreadable OCR scan text")       
    if category == "Invoice":
        if meta.get("total_amount") in ["N/A", "", None]:
            issues.append("Invoice missing monetary total amount")
        if meta.get("document_id") in ["N/A", "", None]:
            issues.append("Invoice missing document/invoice number")           
    elif category == "Agreement":
        if meta.get("vendor_or_parties") in ["N/A", "", None]:
            issues.append("Agreement missing contracting parties")          
    if meta.get("date") in ["N/A", "", None]:
        issues.append("Missing transaction or effective date")       
    if issues:
        anomalies.append({
            "filename": filename,
            "category": category,
            "anomaly_count": len(issues),
            "detected_issues": "; ".join(issues)
        })
anomaly_csv = os.path.join(final_dir, "anomaly_report.csv")
anomaly_json = os.path.join(final_dir, "anomaly_report.json")
df_anomalies = pd.DataFrame(anomalies)
df_anomalies.to_csv(anomaly_csv, index=False)
with open(anomaly_json, "w", encoding="utf-8") as f:
    json.dump(anomalies, f, indent=4)
print(" Automated Data Audit Complete!")
print(f"  Flagged {len(anomalies)} documents with potential data anomalies out of {len(dataset)} total scans.")
print(f"  Anomaly CSV saved to: {anomaly_csv}")
print(f"  Anomaly JSON saved to: {anomaly_json}")

 Structured Deliverables Successfully Exported!
  JSON: project\final_folder\extracted_data.json
  CSV:  project\final_folder\extracted_data.csv

 Automated Data Audit Complete!
  Flagged 24 documents with potential data anomalies out of 100 total scans.
  Anomaly CSV saved to: project\final_folder\anomaly_report.csv
  Anomaly JSON saved to: project\final_folder\anomaly_report.json
